In [ ]:
# Configuration
# Set WORKSPACE_ROOT to a writable absolute path, then select Run All.
WORKSPACE_ROOT = "/path/to/workspace"
AUTO_SETUP = True
RUN_EXPERIMENTS = True
SKIP_COMPLETED = True
RUN_SEEDS = [0, 1, 2, 3]
DEVICE = "cuda"
CREATE_ZIP = False

SESSION = None
REPOSITORY_ROOT = None

def run_experiment(method, seed):
    if SESSION is None:
        raise RuntimeError("Run the setup cells first.")
    SESSION.run_experiment(method, seed)


In [ ]:
# Environment setup and verified Git checkout
import importlib.util
import os
from pathlib import Path
import re
import shutil
import subprocess
import tempfile
import traceback

SOURCE_URL = "https://github.com/ZoyaAnwar/adaptive-k-MIP-GraphGPS.git"
PINNED_COMMIT = None  # Maintainer: set to the full SHA after the study code is published.
REQUIRED_FILES = (
    "graphgps/adaptive_k/train_schedule.py",
    "configs/PascalVOC_NewSchedules/mirrored_exponential.yaml",
    "configs/PascalVOC_NewSchedules/decrease_increase.yaml",
    "study/new_schedules_runner.py",
    "study/new_schedules_worker.py",
    "study/new_schedules_report.py",
)


def git(repo, log, *args):
    env = os.environ.copy()
    for key in ("GIT_DIR", "GIT_WORK_TREE", "GIT_COMMON_DIR", "GIT_INDEX_FILE",
                "GIT_OBJECT_DIRECTORY", "GIT_ALTERNATE_OBJECT_DIRECTORIES"):
        env.pop(key, None)
    env.update(GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="Never")
    process = subprocess.run(
        ["git", "--no-optional-locks", *args], cwd=repo, env=env, stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", timeout=300,
    )
    if process.stdout:
        log.write(process.stdout)
        log.flush()
    if process.returncode:
        raise RuntimeError(f"Git {args[0]} failed with exit code {process.returncode}.")
    return process.stdout.strip()


def verify_checkout(repo, log):
    if (not repo.is_dir() or not (repo / ".git").is_dir()
            or Path(git(repo, log, "rev-parse", "--show-toplevel")).resolve() != repo.resolve()):
        raise RuntimeError(f"Expected a dedicated Git checkout at {repo}.")
    if git(repo, log, "config", "--local", "--get", "remote.origin.url") != SOURCE_URL:
        raise RuntimeError(f"Unexpected Git origin at {repo}.")
    if git(repo, log, "rev-parse", "HEAD") != PINNED_COMMIT:
        raise RuntimeError(f"Checkout at {repo} does not match {PINNED_COMMIT}.")
    if git(repo, log, "status", "--porcelain", "--untracked-files=all"):
        raise RuntimeError(f"Checkout at {repo} has local changes.")
    git(repo, log, "ls-files", "--error-unmatch", "--", *REQUIRED_FILES)
    if any(not (repo / name).is_file() for name in REQUIRED_FILES):
        raise RuntimeError(f"Checkout at {repo} is missing required study files.")


def resolve_repository(workspace_path):
    repository_cache = workspace_path / "repository"
    setup_logs = workspace_path / "setup_logs"
    directories = (repository_cache, workspace_path / "datasets", workspace_path / "results", setup_logs)
    for directory in directories:
        if directory.is_symlink() or (directory.exists() and not directory.is_dir()):
            raise ValueError(f"Use a regular directory for {directory}.")
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(prefix="source-", suffix=".log", dir=setup_logs,
                                     delete=False, mode="w", encoding="utf-8") as log:
        try:
            if not isinstance(PINNED_COMMIT, str) or not re.fullmatch(r"[0-9a-f]{40}", PINNED_COMMIT):
                raise RuntimeError("No study commit is pinned. Obtain the released notebook after "
                                   "the study code has been published to GitHub.")
            if shutil.which("git") is None:
                raise RuntimeError("Install Git on the GPU machine and rerun setup.")
            log.write(f"Repository: {SOURCE_URL}\nPinned commit: {PINNED_COMMIT}\n")
            slot = repository_cache / PINNED_COMMIT
            target = slot / "checkout"
            if not os.path.lexists(slot):
                with tempfile.TemporaryDirectory(prefix="clone-", dir=repository_cache) as temporary:
                    stage = Path(temporary) / "checkout"
                    stage.mkdir()
                    git(stage, log, "init", "--template=")
                    git(stage, log, "remote", "add", "origin", SOURCE_URL)
                    try:
                        git(stage, log, "fetch", "--depth=1", "--no-tags", "origin", PINNED_COMMIT)
                    except RuntimeError as error:
                        raise RuntimeError(f"Could not retrieve {PINNED_COMMIT} from GitHub. "
                                           "Check revision availability and Git credentials.") from error
                    git(stage, log, "checkout", "--detach", "FETCH_HEAD")
                    verify_checkout(stage, log)
                    try:
                        slot.mkdir()
                    except FileExistsError:
                        pass  # Another launch may have published the same revision.
                    else:
                        stage.rename(target)
            if slot.is_symlink() or target.is_symlink():
                raise RuntimeError(f"Unexpected symbolic link in the repository cache at {slot}.")
            verify_checkout(target, log)
            return target
        except (OSError, RuntimeError, subprocess.SubprocessError) as error:
            traceback.print_exc(file=log)
            raise RuntimeError(f"{error} See {log.name}") from error


workspace_path = Path(WORKSPACE_ROOT).expanduser()
if not workspace_path.is_absolute() or WORKSPACE_ROOT == "/path/to/workspace":
    raise ValueError("Set WORKSPACE_ROOT to a writable absolute workspace path in cell 1.")
SESSION = None
REPOSITORY_ROOT = resolve_repository(workspace_path.resolve())
spec = importlib.util.spec_from_file_location("pascalvoc_schedule_runner", REPOSITORY_ROOT / "study/new_schedules_runner.py")
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
session = runner.ExperimentSession(
    WORKSPACE_ROOT, REPOSITORY_ROOT, run_experiments=RUN_EXPERIMENTS,
    skip_completed=SKIP_COMPLETED, seeds=RUN_SEEDS, device=DEVICE,
    create_zip=CREATE_ZIP,
)
session.prepare_environment(auto_setup=AUTO_SETUP)
if session.environment["repository"]["commit"] != PINNED_COMMIT:
    raise RuntimeError("The repository revision changed during environment setup.")
SESSION = session


In [ ]:
# Prepare PascalVOC-SP with the repository loader and standard cache.
SESSION.prepare_dataset()


In [ ]:
# Validate the tuned configurations, k trajectories, output paths and manifest.
SESSION.initialize()


In [ ]:
# Mirrored exponential, seed 0
run_experiment(method="mirrored_exponential", seed=0)


In [ ]:
# Mirrored exponential, seed 1
run_experiment(method="mirrored_exponential", seed=1)


In [ ]:
# Mirrored exponential, seed 2
run_experiment(method="mirrored_exponential", seed=2)


In [ ]:
# Mirrored exponential, seed 3
run_experiment(method="mirrored_exponential", seed=3)


In [ ]:
# Decrease then increase, seed 0
run_experiment(method="decrease_increase", seed=0)


In [ ]:
# Decrease then increase, seed 1
run_experiment(method="decrease_increase", seed=1)


In [ ]:
# Decrease then increase, seed 2
run_experiment(method="decrease_increase", seed=2)


In [ ]:
# Decrease then increase, seed 3
run_experiment(method="decrease_increase", seed=3)


In [ ]:
# Validate and export all available results, including failed and incomplete attempts.
SESSION.export()
